In [1]:
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator
from ta.trend import EMAIndicator, SMAIndicator

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from backtesting import Strategy
from backtesting import Backtest

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
start_date = '2026-1-1 00:00'
end_date = '2026-12-30 00:00'
# start_date = '2022-01-1 00:00'
# end_date = '2026-9-16 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"

In [3]:
# read data
df_h1_raw = pd.read_csv(f"./Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')

df_h1_raw

,Open,High,Low,Close,Tickvol
Date,,,,,
2018-08-07 11:00:00,144.154,144.204,144.088,144.134,5948
2018-08-07 12:00:00,144.133,144.149,144.012,144.062,6595
2018-08-07 13:00:00,144.062,144.075,143.808,143.932,7891
2018-08-07 14:00:00,143.933,143.949,143.815,143.869,7951
2018-08-07 15:00:00,143.869,144.088,143.863,144.087,8342
...,...,...,...,...,...
2026-09-24 10:00:00,209.795,210.067,209.795,210.017,13720
2026-09-24 11:00:00,210.017,210.047,209.813,209.996,13711
2026-09-24 12:00:00,209.997,210.105,209.874,210.005,12834


In [4]:
# process
df_h1 = df_h1_raw.loc[start_date : end_date]

indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1["atr"] = indicator_atr_14.average_true_range()

indicator_ema_50 = EMAIndicator(df_h1['Close'], window=50)
df_h1['EMA_50'] = indicator_ema_50.ema_indicator()

indicator_ema_200 = EMAIndicator(df_h1['Close'], window=200)
df_h1['EMA_200'] = indicator_ema_200.ema_indicator()

indicator_sma_20 = SMAIndicator(df_h1['Close'], window=20)
df_h1['SMA_20'] = indicator_sma_20.sma_indicator()

df_h1


,Open,High,Low,Close,Tickvol,atr,EMA_50,EMA_200,SMA_20
Date,,,,,,,,,
2026-01-01 23:00:00,210.689,211.068,210.574,211.068,1267,0.000000,NaN,NaN,NaN
2026-01-02 00:00:00,211.067,211.347,211.018,211.161,4726,0.000000,NaN,NaN,NaN
2026-01-02 01:00:00,211.162,211.221,211.029,211.069,3370,0.000000,NaN,NaN,NaN
2026-01-02 02:00:00,211.068,211.272,211.014,211.257,4029,0.000000,NaN,NaN,NaN
2026-01-02 03:00:00,211.258,211.341,211.198,211.330,2082,0.000000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2026-09-24 10:00:00,209.795,210.067,209.795,210.017,13720,0.285732,209.720696,209.698042,209.50290
2026-09-24 11:00:00,210.017,210.047,209.813,209.996,13711,0.282037,209.731493,209.701007,209.51125
2026-09-24 12:00:00,209.997,210.105,209.874,210.005,12834,0.278392,209.742218,209.704031,209.52340


In [5]:
df_h1.describe()

,Open,High,Low,Close,Tickvol,atr,EMA_50,EMA_200,SMA_20
count,4486.000000,4486.000000,4486.000000,4486.000000,4486.000000,4486.000000,4437.000000,4287.000000,4467.000000
mean,213.252153,213.375346,213.122829,213.254178,9510.555506,0.253502,213.285206,213.377358,213.266202
std,2.635633,2.611592,2.657668,2.633900,6150.234463,0.100248,2.533090,2.227800,2.607551
min,207.298000,207.550000,207.093000,207.300000,416.000000,0.000000,208.105460,209.186930,207.764850
25%,211.375500,211.512000,211.257500,211.386250,5278.500000,0.188991,211.586612,211.779943,211.429700
50%,213.222000,213.340000,213.063500,213.225000,7907.000000,0.228051,213.240330,213.603454,213.203050
75%,215.147500,215.240000,215.043000,215.147500,12285.750000,0.290912,215.020240,214.701439,215.146775
max,219.568000,219.604000,219.536000,219.567000,54492.000000,1.113730,218.618902,218.074462,219.338400


In [6]:
def apply_total_signal(df, ema50_window = 15):
    # Initialize the 'TotalSignal' column
    df_signal_list = []

    for i in range(0, len(df)):
        # trending conditions
        c_long_ema50_ema200 = df['EMA_50'].iloc[i] > df['EMA_200'].iloc[i]
        c_15_candles_above_ema50 = (df['Low'].iloc[i-ema50_window:i] > df['EMA_50'].iloc[i]).all()

        # Previous candle conditions
        candle_low_below_sma20 = df['Low'].iloc[i] < df['SMA_20'].iloc[i]

        # Current candle conditions
        closes_above_prev_high = df['Close'].iloc[i] > df['High'].iloc[i-1]

        signal = 0

        # Combine conditions
        if (c_long_ema50_ema200 and
            c_15_candles_above_ema50 and
            candle_low_below_sma20 and
            closes_above_prev_high):
            signal = 2

        # trending conditions
        c_short_ema50_ema200 = df['EMA_50'].iloc[i] < df['EMA_200'].iloc[i]
        c_15_candles_below_ema50 = (df['High'].iloc[i-ema50_window:i] < df['EMA_50'].iloc[i]).all()

        # Previous candle conditions
        candle_high_above_sma20 = df['High'].iloc[i] > df['SMA_20'].iloc[i]

        # Current candle conditions
        closes_below_prev_low = df['Close'].iloc[i] < df['Low'].iloc[i-1]

        # Combine conditions
        if (c_short_ema50_ema200 and
            c_15_candles_below_ema50 and
            candle_high_above_sma20 and
            closes_below_prev_low):
            signal = 1

        df_signal_list.append(signal)

    return df_signal_list

df_h1['TotalSignal'] = apply_total_signal(df=df_h1, ema50_window=15)

print(df_h1)

                        Open     High      Low    Close  Tickvol       atr  \
Date                                                                         
2026-01-01 23:00:00  210.689  211.068  210.574  211.068     1267  0.000000   
2026-01-02 00:00:00  211.067  211.347  211.018  211.161     4726  0.000000   
2026-01-02 01:00:00  211.162  211.221  211.029  211.069     3370  0.000000   
2026-01-02 02:00:00  211.068  211.272  211.014  211.257     4029  0.000000   
2026-01-02 03:00:00  211.258  211.341  211.198  211.330     2082  0.000000   
...                      ...      ...      ...      ...      ...       ...   
2026-09-24 10:00:00  209.795  210.067  209.795  210.017    13720  0.285732   
2026-09-24 11:00:00  210.017  210.047  209.813  209.996    13711  0.282037   
2026-09-24 12:00:00  209.997  210.105  209.874  210.005    12834  0.278392   
2026-09-24 13:00:00  210.004  210.015  209.890  209.963    15622  0.267435   
2026-09-24 14:00:00  209.965  210.073  209.846  209.848    19089

In [7]:
len(df_h1[df_h1.TotalSignal != 0])

67

In [8]:
print(df_h1.loc[start_date : end_date])

def pointpos(x):
    if x['TotalSignal']==2:
        return x['Low']-1e-4
    elif x['TotalSignal']==1:
        return x['High']+1e-4
    else:
        return np.nan

df_h1['pointpos'] = df_h1.apply(lambda row: pointpos(row), axis=1)

                        Open     High      Low    Close  Tickvol       atr  \
Date                                                                         
2026-01-01 23:00:00  210.689  211.068  210.574  211.068     1267  0.000000   
2026-01-02 00:00:00  211.067  211.347  211.018  211.161     4726  0.000000   
2026-01-02 01:00:00  211.162  211.221  211.029  211.069     3370  0.000000   
2026-01-02 02:00:00  211.068  211.272  211.014  211.257     4029  0.000000   
2026-01-02 03:00:00  211.258  211.341  211.198  211.330     2082  0.000000   
...                      ...      ...      ...      ...      ...       ...   
2026-09-24 10:00:00  209.795  210.067  209.795  210.017    13720  0.285732   
2026-09-24 11:00:00  210.017  210.047  209.813  209.996    13711  0.282037   
2026-09-24 12:00:00  209.997  210.105  209.874  210.005    12834  0.278392   
2026-09-24 13:00:00  210.004  210.015  209.890  209.963    15622  0.267435   
2026-09-24 14:00:00  209.965  210.073  209.846  209.848    19089

In [9]:
start_date_chart = '2026-08-01 00:00:00'
end_date_chart = '2026-08-30 00:00:00'
dfpl = df_h1.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")
number_trades = len(dfpl[dfpl.TotalSignal != 0])

print(number_trades)
# Create a plot with 2 rows
fig = make_subplots(rows=2, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

# Add Bollinger Bands, EMA lines on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['EMA_50'],
                         line=dict(color='blue', width=1),
                         name="EMA_50"),
              row=1, col=1)
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['EMA_200'],
                         line=dict(color='yellow', width=1),
                         name="EMA_200"),
              row=1, col=1)
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['SMA_20'],
                         line=dict(color='green', width=1),
                         name="SMA_20"),
              row=1, col=1)

# Add markers for trade entry points on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['pointpos'], mode="markers",
                         marker=dict(size=8, color="MediumPurple"),
                         name="entry"),
              row=1, col=1)

fig.update_layout(width=1200, height=800, sliders=[])
fig.show()

11


In [10]:
dfopt = df_h1
def SIGNAL():
    return dfopt.TotalSignal

class MyStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 1
    start_trade_hour = 5
    stop_trade_hour = 20
    
    def init(self):
        super().init()
        self.signal1 = self.I(SIGNAL)

    def next(self):
        super().next()
        c_time_trade = self.start_trade_hour < self.data.index[-1].hour and self.data.index[-1].hour < self.stop_trade_hour

        sl = abs(self.data.Close[-1] - self.data.Low[-1])
        tp = self.TPcoef * sl
        c_sl_tp = sl < self.data.atr[-1] * 2.5 and sl > self.data.atr[-1] * 1

        if len(self.trades)==0 and c_time_trade and c_sl_tp:
            if self.signal1==2:
                sl1 = self.data.Close[-1] - sl
                tp1 = self.data.Close[-1] + tp
                self.buy(sl=sl1, tp=tp1, size=self.mysize)

            if self.signal1==1:
                sl1 = self.data.Close[-1] + sl
                tp1 = self.data.Close[-1] - tp
                self.sell(sl=sl1, tp=tp1, size=self.mysize)

In [12]:
bt = Backtest(df_h1, MyStrat, cash=10000, margin=1/10, commission=0.001) #0.0002
stats = bt.run()

print(stats)

bt.plot()

Start                     2026-01-01 23:00:00
End                       2026-09-24 14:00:00
Duration                    265 days 15:00:00
Exposure Time [%]                     0.31208
Equity Final [$]                   9316.72211
Equity Peak [$]                       10000.0
Commissions [$]                     683.17589
Return [%]                           -6.83278
Buy & Hold Return [%]                -0.57801
Return (Ann.) [%]                     -8.9598
Volatility (Ann.) [%]                 3.24602
CAGR [%]                             -9.27332
Sharpe Ratio                         -2.76024
Sortino Ratio                        -2.48757
Calmar Ratio                          -1.3113
Alpha [%]                            -6.83436
Beta                                 -0.00274
Max. Drawdown [%]                    -6.83278
Avg. Drawdown [%]                    -6.83278
Max. Drawdown Duration      164 days 06:00:00
Avg. Drawdown Duration      164 days 06:00:00
# Trades                          

GridPlot(id='p1639', ...)